# RDD Baseline — Naive Bayes Classifier

Unoptimised MapReduce implementation using the PySpark RDD API.
Uses `flatMap` + `groupByKey` — intentionally inefficient to establish a performance lower bound.
See `naive_bayes_rdd_optimized.ipynb` for the improved version.


In [1]:
import time
import math
import sys
import os

from pyspark.sql import SparkSession

project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

from core.naive_bayes import compute_log_probs, evaluate
from data.loader import load_car_rdd

# predict is defined locally to avoid serialization issues with Spark when using a global function.
def predict(log_prob_table, test_point):
    best_class = None
    best_score = float('-inf')
    for class_label in log_prob_table['classes']:
        score = log_prob_table['log_class_probs'][class_label]
        for feat_idx, value in enumerate(test_point):
            key = f'feat_{feat_idx}_{value}_{class_label}'
            if key in log_prob_table['log_feature_probs']:
                score += log_prob_table['log_feature_probs'][key]
            else:
                score += log_prob_table['fallback_log_probs'].get(
                    (feat_idx, class_label), math.log(1e-10)
                )
        if score > best_score:
            best_score = score
            best_class = class_label
    return best_class

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('NaiveBayes-RDD-Baseline')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')
print('SparkSession ready.')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 19:43:52 WARN Utils: Your hostname, Alis-Macbook-Pro.local, resolves to a loopback address: 127.0.0.1; using 129.104.242.145 instead (on interface en0)
26/04/02 19:43:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 19:43:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession ready.


In [2]:
# Cell 2 — Load data
DATA_PATH = None

train_rdd, test_rdd = load_car_rdd(spark, filepath=DATA_PATH)

# Print basic stats
train_count = train_rdd.count()
test_count  = test_rdd.count()
print(f'Train rows : {train_count}')
print(f'Test rows  : {test_count}')
print(f'Sample row : {train_rdd.first()}')

class_dist = train_rdd.map(lambda row: row[0]).countByValue()
print(f'Class distribution: {dict(class_dist)}')

Train rows : 4
Test rows  : 1
Sample row : ('unacc', ['low', 'med', '2', '2', 'small', 'low'])
Class distribution: {'unacc': 2, 'acc': 2}


In [ ]:
# MapReduce

t_train_start = time.time()

# Map phase — mapRow
def map_row(row):
    label, features = row
    pairs = []
    for feat_idx, value in enumerate(features):
        pairs.append((f'feat_{feat_idx}_{value}_{label}', 1))
    pairs.append((f'class_{label}', 1))
    return pairs

# flatMap applies map_row to every row and flattens the list-of-lists.
mapped_rdd = train_rdd.flatMap(map_row)

# REDUCE PHASE — groupByKey 
grouped_rdd = mapped_rdd.groupByKey()
counts_rdd = grouped_rdd.mapValues(sum)

# Collect all counts to the driver.
all_counts = dict(counts_rdd.collect())

train_time = time.time() - t_train_start
print(f'[TIMING] Training: {train_time:.4f}s')
print(f'Total unique keys : {len(all_counts)}')
print(f'Sample counts     : {list(all_counts.items())[:4]}')

[TIMING] Training: 0.1779s
Total unique keys : 17
Sample counts     : [('feat_0_low_unacc', 2), ('feat_0_low_acc', 2), ('class_acc', 2), ('feat_2_2_unacc', 2)]


In [4]:
# Separate all_counts into the two input dicts compute_log_probs() expects.
class_counts   = {k[6:]: v for k, v in all_counts.items() if k.startswith('class_')}
feature_counts = {k: v       for k, v in all_counts.items() if k.startswith('feat_')}
class_totals   = class_counts  # same dict for standard Naive Bayes

log_prob_table = compute_log_probs(class_counts, feature_counts, class_totals)

# Log-probabilities should all be negative (all P < 1).
print(f"Classes           : {log_prob_table['classes']}")
print(f"Log class probs   : {log_prob_table['log_class_probs']}")
print(f"Sample feat probs : {list(log_prob_table['log_feature_probs'].items())[:3]}")

Classes           : ['acc', 'unacc']
Log class probs   : {'acc': -0.6931471805599453, 'unacc': -0.6931471805599453}
Sample feat probs : [('feat_0_low_unacc', 0.0), ('feat_0_low_acc', 0.0), ('feat_2_2_unacc', 0.0)]


In [5]:
t_pred_start = time.time()

predictions_rdd = test_rdd.map(
    lambda row: (row[0], predict(log_prob_table, row[1]))
)
predictions_list = predictions_rdd.collect()

pred_time = time.time() - t_pred_start
print(f'[TIMING] Prediction: {pred_time:.4f}s')
print(f'Sample predictions : {predictions_list[:5]}')

[TIMING] Prediction: 0.0973s
Sample predictions : [('good', 'acc')]


In [ ]:
# Evaluation

true_labels  = [pair[0] for pair in predictions_list]
pred_labels  = [pair[1] for pair in predictions_list]

accuracy = evaluate(pred_labels, true_labels)

print(f'Accuracy     : {accuracy:.4f} ({accuracy * 100:.2f}%)')
print(f'Train time   : {train_time:.4f}s')
print(f'Predict time : {pred_time:.4f}s')
print(f'Total time   : {train_time + pred_time:.4f}s')


Accuracy     : 0.0000 (0.00%)
Train time   : 0.1779s
Predict time : 0.0973s
Total time   : 0.2752s
